# 🧨 Lab 02 — The column had a costume change
Run Lab 01 first. Run this as a job with `feed_table=zops_feed_v1` for a clean control. Scope ZeroOps to this job and the disposable lab schema. Run it again with `feed_table=zops_feed_v2`. The publish cell should fail with `UNRESOLVED_COLUMN` for `amount_cents`. Save the two run IDs and the failing task output.

This exercise simulates a breaking upstream rollout by switching input tables. Both revisions stay available for inspection. Let ZeroOps investigate the failed run before pressing Retry. Use the job run history as ground truth if an insight is still pending.


In [ ]:
%sql
SELECT assert_true(:catalog RLIKE '^[A-Za-z_][A-Za-z0-9_]*$' AND :schema RLIKE '^zeroops_lab_[a-z0-9_]+$', 'Use a dedicated zeroops_lab_<alias> schema'),
       assert_true(:feed_table IN ('zops_feed_v1', 'zops_feed_v2'), 'Use only the two lab fixtures');


In [ ]:
%sql
DESCRIBE TABLE IDENTIFIER(:catalog || '.' || :schema || '.' || :feed_table);


## Review the proposed adapter
For a v2-only consumer, try `SUM(amount_minor) AS revenue_cents` in a scratch copy. Expect three purchases and 2,500 USD cents. A rollback to v1 restores the control but leaves v2 unsupported. A migration that must accept both revisions needs an explicit version-aware projection before the aggregate. Referencing both missing/present fields inside `COALESCE` still requires both names to resolve. Verify the currency and units before accepting any mapping.

Record whether ZeroOps named the upstream change and cited evidence from the failing task or schema. Grade the proposed change using the control rather than the confidence of the summary. The publish cell deliberately retains the old contract. Check output table history before showing the dashboard a green screenshot from yesterday.


In [ ]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :schema || '.zops_daily_revenue') USING DELTA AS
SELECT currency, COUNT(*) AS purchases, SUM(amount_cents) AS revenue_cents
FROM IDENTIFIER(:catalog || '.' || :schema || '.' || :feed_table)
GROUP BY currency;


In [ ]:
%sql
SELECT currency, purchases, revenue_cents
FROM IDENTIFIER(:catalog || '.' || :schema || '.zops_daily_revenue');
